In [ ]:
!pip install transformers evaluate accelerate
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import requests
import zipfile
import io

import re
import nltk
import torch
import evaluate
import numpy as np
import pandas as pd

from math import inf
from datasets import Dataset
from nltk.corpus import words
from nltk.util import everygrams
from nltk.lm import MLE, Laplace
from nltk import bigrams, trigrams
from sklearn.model_selection import train_test_split
from nltk.lm.preprocessing import pad_both_ends, flatten, padded_everygram_pipeline
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
# Download authorship training files
url = "https://jimtmooney.github.io/Courses/S25/hw/ngram_authorship_train.zip"
response = requests.get(url)
file = zipfile.ZipFile(io.BytesIO(response.content))
file.extractall()

# Download punkt_tab
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
def preprocess_file(filepath):
  f = open(filepath, "r")
  text = f.readlines()
  f.close()
  processed_text = [] # Create a new list to store processed lines
  for line in text:
    tokens = nltk.word_tokenize(line)
    tokens = [word.lower() for word in tokens]

    tokens = [re.sub(r'[^\w\s]', '', word) for word in tokens] # Adjusted to update tokens
    tokens = [word for word in tokens if word.isalpha()]
    if tokens:
      processed_text.append(tokens)
  return processed_text

In [27]:
# Prepare environment
SEED = 42
N = 5
authors = ["austen", "dickens", "tolstoy", "wilde"]
folder_path = 'ngram_authorship_train/'

In [ ]:
# Retrieve texts from stored files
# This is returned as a dictionary
# Each entry is structured Author: Text
def retrieve_texts(authors):
  texts = {}
  for author in authors:
    texts[author] = preprocess_file(folder_path + author + '_utf8.txt')
  return texts


In [ ]:
# GENERATIVE MODELS

# partitions data into train/test splits.
# small: reduces datasets to 100 to reduce train time
# Returns a dictionary structured as Author: (train_data, test_data)
def partition_data(data, is_small):
  data = data
  if is_small:
    data = {author: text[:100] for author, text in data.items()}

  # List of tuples structured as author_train, author_test
  data_split = {}
  for author, text in data.items():
    train, test = train_test_split(text, test_size=0.1, random_state=SEED)
    data_split[author] = (train, test)

  return data_split

# Trains Laplace models for each author
# Returns a dictionary structured as Author: Model
def train_models(n, data):
  author_lms = {}
  for author, (train, test) in data.items():
    author_lm = Laplace(n)
    train_data, padded_sents = padded_everygram_pipeline(n, train)
    author_lm.fit(train_data, padded_sents)
    author_lms[author] = author_lm
  return author_lms

# using bigrams only to avoid inf perplexity issue
# Returns the name of the author with the minimum perplexity for the given text
def classify(models, text):
  text = list(pad_both_ends(text, n=N))
  text = list(bigrams(text))

  #print(f"Classifying: {text}")
  min_perplexity = np.inf
  min_author = "unknown"

  # print(text)
  if not text:
    print("empty text")
    return min_author

  for author, model in models.items():
    a_text = model.vocab.lookup(text)
    if a_text[0] == ('<s>', '<UNK>'): a_text = a_text[1:]
    if a_text[-1] == ('<UNK>', '</s>'): a_text = a_text[:-1]
    if a_text and ('<UNK>', '<UNK>') not in a_text:
      perplexity = model.perplexity(a_text)
      #print(f"Perplexity: {perplexity}")
      if perplexity < min_perplexity:
        min_perplexity = perplexity
        min_author = author

  return min_author

# Evaluates a model by counting the number of correct classifications against total
def test_model(author_data, models):
  author, data = author_data
  print(f"Now classifying {author}'s texts:")
  total = 0
  correct = 0
  for text in data[author][1]:
    total += 1
    if classify(models, text) == author:
      correct += 1
  return correct/total


In [28]:
# Prepare data for execution
data = retrieve_texts(authors)
data_split = partition_data(data, True)
models = train_models(N, data_split)

In [29]:
# Potentially useful debug output
for model in models.values():
  print(model.counts['golden'])
  print(model.counts[['golden']]['hair'])
debug_text = data["wilde"][0][1]
print(debug_text)
print(classify(models, debug_text))

test_text = list(pad_both_ends(debug_text, N))
test_text = list(everygrams(test_text, max_len=N))
display(models['austen'].perplexity(test_text))

0
0
0
0
0
0
1
0
if
dickens


37.629941690973126

In [ ]:
prompts = ["Write a short love poem.",
           "What was one time you showed leadership skills?",
           "Give a definition of beauty",
           "What is the average airspeed of a swallow",
           "Write a recipe for chocolate cake"]

In [30]:
for prompt in prompts:
  print(f'Giving models the following prompt: {prompt}\n---------------')
  for author in authors:
    model = models[author]
    text = model.generate(num_words=50, text_seed=prompt)
    print(f'{author}: {" ".join(text)}')
    print(f'Perplexity: {model.perplexity(text)}')
  print()

Giving models the following prompt: Write a short love poem.
---------------
austen: duty not by manoeuvring and finessing but by vigour and </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s>
Perplexity: 406.9999999999991
dickens: the object that </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s>
Perplexity: 380.0000000000012
tolstoy: themselves cooked their porridge for the first time </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s> </s>
Perplexity: 403.00000000000153
wilde: basil hallward s compliments h

#Part 2 - Discriminative Authorship Classifier

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess_text_part2(filepath):
  f = open(filepath, "r")
  text = f.readlines()
  f.close()
  for index, line in enumerate(text):
      text[index] = line.strip()
  return text

def organize_text_data(author):
  lines = preprocess_text_part2(folder_path + author + '_utf8.txt')
  df = pd.DataFrame(columns=('text', 'label'))
  i = 0
  for line in lines:
    df.loc[i] = [line, author]
    i = i + 1
  return df

def tokenize_data_part2(examples):
   return tokenizer(examples, truncation = True)

In [ ]:
id2label = {i: label for i, label in enumerate(authors)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
# Prepare each author dataframe, and create a concatenated aggregate dataframe
author_dfs = {}
for author in authors:
  author_dfs[author] = organize_text_data(author)
author_df = pd.concat(author_dfs.values())

In [ ]:
display(author_df)

,text,label
0,family may be. We ought to be acquainted with ...,austen
1,"Churchill’s temper, before we pretend to decid...",austen
2,"can do. He may, at times, be able to do a grea...",austen
3,at others.”,austen
4,,austen
...,...,...
9995,As I had said some very unjust and bitter thin...,wilde
9996,"I determined to go and see him at once, and to...",wilde
9997,"for my behaviour. Accordingly, the next morni...",wilde
9998,"Walk, and found Erskine sitting in his library...",wilde


In [ ]:
# Tokenize and split data
author_df['input_ids'] = author_df['text'].apply(lambda x: tokenize_data_part2(x)['input_ids'])
author_df['label'] = author_df['label'].map(label2id)
display(author_df)
train_df, test_df = train_test_split(author_df, test_size=0.1, random_state=SEED)
train = Dataset.from_pandas(train_df)
test = Dataset.from_pandas(test_df)

,text,label,input_ids
0,family may be. We ought to be acquainted with ...,0,"[101, 2155, 2089, 2022, 1012, 2057, 11276, 200..."
1,"Churchill’s temper, before we pretend to decid...",0,"[101, 10888, 1521, 1055, 12178, 1010, 2077, 20..."
2,"can do. He may, at times, be able to do a grea...",0,"[101, 2064, 2079, 1012, 2002, 2089, 1010, 2012..."
3,at others.”,0,"[101, 2012, 2500, 1012, 1524, 102]"
4,,0,"[101, 102]"
...,...,...,...
9995,As I had said some very unjust and bitter thin...,3,"[101, 2004, 1045, 2018, 2056, 2070, 2200, 4895..."
9996,"I determined to go and see him at once, and to...",3,"[101, 1045, 4340, 2000, 2175, 1998, 2156, 2032..."
9997,"for my behaviour. Accordingly, the next morni...",3,"[101, 2005, 2026, 9164, 1012, 11914, 1010, 199..."
9998,"Walk, and found Erskine sitting in his library...",3,"[101, 3328, 1010, 1998, 2179, 27139, 3564, 199..."


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # Not the part2 tokenizer?
# Proportion of correct predictions among the total number of cases processed
accuracy = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(authors), id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir='CSCI5541_Homework3',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=N,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

<ipython-input-18-1d8fb14cc5a2>:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hawfi004 (hawfi004-university-of-minnesota) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.754500,0.736348,0.690179
2,0.610800,0.702910,0.710536


TrainOutput(global_step=3150, training_loss=0.7435143025716145, metrics={'train_runtime': 1499.9782, 'train_samples_per_second': 67.201, 'train_steps_per_second': 2.1, 'total_flos': 626783572321536.0, 'train_loss': 0.7435143025716145, 'epoch': 2.0})